In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
from scipy import stats

from paths import paths, prepare_preprocessing

In [ ]:
prepare_preprocessing()

In [ ]:
# Load all relevant data
walkability_by_duration = {
    minutes: gpd.read_parquet(paths.preprocessing.walkability[minutes].slim) for minutes in (10, 15, 20)
}
walkability_enriched_by_duration = {
    minutes: gpd.read_parquet(paths.preprocessing.walkability[minutes].enriched) for minutes in (10, 15, 20)
}

lsoa_ids = set(walkability_by_duration[15].LSOA21CD)

qof = pd.read_csv(paths.preprocessing.health.prevalence, index_col = 0)

medsat = (
    pd.read_csv(paths.raw.shared.medsat, index_col = 0)
    .reset_index()
    .rename(columns = {'geography code': 'LSOA21CD'})
    .set_index('LSOA21CD')
)

imd = pd.read_csv(paths.preprocessing.context.imd, index_col = 0)
pollution = pd.read_csv(paths.preprocessing.context.pollution, index_col = 0)
ndvi = pd.read_csv(paths.preprocessing.context.ndvi, index_col = 0)

boroughs = (
    pd.read_csv(paths.raw.shared.lsoa_lookup, usecols = ['LSOA21CD', 'LAD22NM'])
    .drop_duplicates()
    .set_index('LSOA21CD', verify_integrity = True)
)

In [ ]:
# Extract the covariate columns from the MEDSAT dataset
RACE_COLS = ['c_percent asian', 'c_percent black', 'c_percent mixed', 'c_percent white']

AGE_BANDS = {
    'pct_0_14': ['c_percent Aged 4 years and under', 'c_percent Aged 5 to 9 years', 'c_percent Aged 10 to 14 years'],
    'pct_15_44': [
        'c_percent Aged 15 to 19 years',
        'c_percent Aged 20 to 24 years',
        'c_percent Aged 25 to 29 years',
        'c_percent Aged 30 to 34 years',
        'c_percent Aged 35 to 39 years',
        'c_percent Aged 40 to 44 years'
    ],
    'pct_45_64': [
        'c_percent Aged 45 to 49 years',
        'c_percent Aged 50 to 54 years',
        'c_percent Aged 55 to 59 years',
        'c_percent Aged 60 to 64 years'
    ],
    'pct_65p': [
        'c_percent Aged 65 to 69 years',
        'c_percent Aged 70 to 74 years',
        'c_percent Aged 75 to 79 years',
        'c_percent Aged 80 to 84 years',
        'c_percent Aged 85 years and over'
    ]
}

SEX_COLS = ['c_percent male']

cov_medsat = medsat[RACE_COLS + SEX_COLS + sum(AGE_BANDS.values(), [])].copy()

for new, olds in AGE_BANDS.items():
    cov_medsat[new] = cov_medsat[olds].sum(axis = 1)

cov_medsat.drop(columns = sum(AGE_BANDS.values(), []), inplace = True)

In [ ]:
# Merge covariates with IMD and pollution data
covariate_columns = {
    'c_percent male': 'pct_male',
    'c_percent asian': 'pct_asian',
    'c_percent black': 'pct_black',
    'c_percent mixed': 'pct_mixed',
    'c_percent white': 'pct_white'
}

covariates = cov_medsat.join(imd[['imd_score']]).join(pollution[['nox']]).join(ndvi[['ndvi']])
covariates = covariates.rename(columns = covariate_columns)

In [ ]:
# Standardise the covariates
z_cols = [
    'pct_0_14',
    'pct_45_64',
    'pct_65p',
    'pct_male',
    'pct_black',
    'pct_asian',
    'pct_mixed',
    'nox',
    'ndvi',
    'imd_score'
]

for col in z_cols:
    covariates[f'z_{col}'] = stats.zscore(covariates[col], nan_policy = 'omit')

In [ ]:
# Merge health and covariate values into each walkshed dataset
slim_by_duration = {
    minutes: frame.set_index('LSOA21CD', drop = False).join(qof.add_prefix('qof_')).join(covariates)
    for minutes, frame in walkability_by_duration.items()
}
enriched_by_duration = {
    minutes: frame.set_index('LSOA21CD', drop = False).join(qof.add_prefix('qof_')).join(covariates)
    for minutes, frame in walkability_enriched_by_duration.items()
}

In [ ]:
# Set outlier QOF values to NaN
def winsorize_high_to_nan(df, cols, pct = 0.001):
    hi_q = 1 - pct

    for c in cols:
        hi = df[c].quantile(hi_q)
        df.loc[df[c] > hi, c] = np.nan

CONDITION_COLS = [c for c in slim_by_duration[15].columns if c.startswith('qof_')]

for frame in [*slim_by_duration.values(), *enriched_by_duration.values()]:
    winsorize_high_to_nan(frame, CONDITION_COLS)

In [ ]:
# Add unknown ethnicity percentages and borough names
for frame in [*slim_by_duration.values(), *enriched_by_duration.values()]:
    frame['pct_unknown'] = 1 - frame[['pct_white', 'pct_black', 'pct_asian', 'pct_mixed']].sum(axis = 1)
    frame['LAD22NM'] = boroughs.LAD22NM.reindex(frame.index)

In [ ]:
for minutes in (10, 15, 20):
    slim_by_duration[minutes].to_parquet(
        paths.results.by_duration[minutes], index = True, compression = 'gzip'
    )
    enriched_by_duration[minutes].to_parquet(
        paths.results.enriched_by_duration[minutes], index = True, compression = 'gzip'
    )